# 📝 Exercício M1.05

O objetivo deste exercício é avaliar o impacto do pré-processamento de features
em um pipeline que usa um classificador baseado em árvore de decisão em vez de
uma regressão logística.

* A primeira pergunta é avaliar empiricamente se escalonar as features
  numéricas é útil ou não;
* A segunda pergunta é avaliar se é empiricamente melhor (tanto do ponto de
  vista computacional quanto estatístico) usar categorias codificadas em
  inteiros ou codificadas em one-hot.

In [ ]:
import pandas as pd

adult_census = pd.read_csv("../datasets/adult-census.csv")

In [ ]:
target_name = "class"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name, "education-num"])

Como nos notebooks anteriores, usamos o utilitário `make_column_selector` para
selecionar apenas colunas com um tipo de dado específico. Além disso, listamos
antecipadamente todas as categorias das colunas categóricas.

In [ ]:
from sklearn.compose import make_column_selector as selector

numerical_columns_selector = selector(dtype_exclude=object)
categorical_columns_selector = selector(dtype_include=object)
numerical_columns = numerical_columns_selector(data)
categorical_columns = categorical_columns_selector(data)

## Pipeline de referência (sem escalonamento numérico e com categorias codificadas em inteiros)

Primeiro, vamos cronometrar o pipeline que usamos no notebook principal para
servir de referência:

In [ ]:
import time

from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)
preprocessor = make_column_transformer(
    (categorical_preprocessor, categorical_columns),
    remainder="passthrough",
)


model = make_pipeline(preprocessor, HistGradientBoostingClassifier())

start = time.time()
cv_results = cross_validate(model, data, target)
elapsed_time = time.time() - start

scores = cv_results["test_score"]

print(
    "A acurácia média da validação cruzada é: "
    f"{scores.mean():.3f} ± {scores.std():.3f} "
    f"com um tempo de ajuste de {elapsed_time:.3f} segundos"
)

## Escalonando features numéricas

Vamos escrever um pipeline semelhante que também escalona as features numéricas
usando `StandardScaler` (ou similar):

In [ ]:
# Escreva seu código aqui.

## Codificação one-hot de variáveis categóricas

Observamos que a codificação em inteiros de variáveis categóricas pode ser
muito prejudicial para modelos lineares. No entanto, isso não parece ser o caso
para modelos `HistGradientBoostingClassifier`, já que o score de validação
cruzada do pipeline de referência com `OrdinalEncoder` é razoavelmente bom.

Vamos ver se conseguimos uma acurácia ainda melhor com `OneHotEncoder`.

Dica: o `HistGradientBoostingClassifier` ainda não suporta dados de entrada
esparsos. Você pode usar `OneHotEncoder(handle_unknown="ignore",
sparse_output=False)` para forçar o uso de uma representação densa como solução
alternativa.

In [ ]:
# Escreva seu código aqui.

## Qual codificador devo usar?

|                          | Ordem significativa           | Ordem não significativa                    |
| ------------------------ | ------------------------------ | ------------------------------------------- |
| Modelo baseado em árvore | `OrdinalEncoder`               | `OrdinalEncoder` com profundidade razoável   |
| Modelo linear            | `OrdinalEncoder` com cautela   | `OneHotEncoder`                              |

<div class="admonition important alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Importante</p>
<ul class="last simple">
<li><tt class="docutils literal">OneHotEncoder</tt>: sempre faz algo
significativo, mas pode ser desnecessariamente lento com árvores.</li>
<li><tt class="docutils literal">OrdinalEncoder</tt>: pode ser prejudicial
para modelos lineares, a menos que sua categoria tenha uma ordem
significativa e você garanta que o <tt class="docutils literal">OrdinalEncoder</tt>
respeite essa ordem. Árvores conseguem lidar bem com o
<tt class="docutils literal">OrdinalEncoder</tt>, desde que sejam
suficientemente profundas. No entanto, quando você permite que a árvore de
decisão cresça muito profundamente, ela pode sofrer overfitting em outras
features.</li>
</ul>
</div>

Além da codificação one-hot e da codificação ordinal de features categóricas, o
scikit-learn oferece o
[`TargetEncoder`](https://scikit-learn.org/stable/modules/preprocessing.html#target-encoder).
Esse codificador é bem adequado para features categóricas nominais com alta
cardinalidade. Essa estratégia de codificação está fora do escopo deste curso,
mas o leitor interessado é encorajado a explorar esse codificador.